# 04 · Causal Pricing in a Two-Sided Marketplace

We now bring it together for a **two-sided marketplace** (riders ↔ drivers, guests ↔ hosts). Two new realities appear that single-market pricing ignores:

1. **Interference / SUTVA violation.** In a marketplace, units are *not* independent. If you give one rider a cheaper price and they book, they consume a driver who is now unavailable to a control-group rider. A naive user-level A/B test therefore **over-states** the true marketplace effect — the treated group's gain is partly *stolen* from controls. This is the central message of Uber's [*Causally-Informed Marketplace Optimization*](https://arxiv.org/html/2407.19078v1).
2. **Heterogeneous price sensitivity.** A commuter at 8am and a leisure rider on Sunday have very different elasticities. The revenue/welfare-optimal price is *personalized*, which means we need a **CATE** (conditional average treatment effect), not one global number.

We use [**EconML**](https://econml.azurewebsites.net/) to recover heterogeneous causal effects of surge, and close with how to assemble a causal pricing structure.

**References**
- [Uber — Practical Marketplace Optimization with Causally-Informed ML (arXiv:2407.19078)](https://arxiv.org/html/2407.19078v1)
- [Cohen et al. — Using Big Data to Estimate Consumer Surplus: Uber (NBER w22627, PDF)](https://www.nber.org/system/files/working_papers/w22627/w22627.pdf)
- [EconML — heterogeneous treatment effects](https://econml.azurewebsites.net/)
- [Airbnb Engineering — Experiments & interference](https://medium.com/airbnb-engineering/experiments-at-airbnb-e2db3abf39e7)

In [ ]:
import sys, os
sys.path.append(os.path.abspath('utils'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datagen import simulate_two_sided

df, gt = simulate_two_sided(n=8000, base_elasticity=-1.2, seed=11)
print('True ATE of surge on booking:', round(gt['ate'], 3))
df.head()

## 1. The observational trap, again

Naively correlate `booked` with `surge`. A confounder (`demand_intensity`) raises *both* surge and booking, so the raw association makes surge look *harmless* (or even good) for bookings — masking that surge genuinely suppresses demand.

In [ ]:
import statsmodels.api as sm
naive = sm.OLS(df['booked'], sm.add_constant(df['surge'])).fit().params['surge']
adj   = sm.OLS(df['booked'], sm.add_constant(df[['surge','demand_intensity']])).fit().params['surge']
print(f'Naive surge→booking slope          : {naive:+.3f}   (confounded, too optimistic)')
print(f'Adjusted for demand_intensity      : {adj:+.3f}')
print(f'TRUE average effect (ATE)          : {gt["ate"]:+.3f}')

## 2. Interference: why a naive A/B test over-states the effect

We simulate a user-level experiment where treated riders see a discount. Because treated bookings consume shared driver supply, the measured lift is larger than the true marketplace lift. A **market-level / switchback** design (randomize the *whole market*, not the user) avoids the leakage.

In [ ]:
rng = np.random.default_rng(0)
N = len(df)
treat = rng.binomial(1, 0.5, size=N)               # user-level randomization
true_user_effect = 0.05                              # each treated rider +5pp on their own
supply = 0.45 * N                                    # finite drivers this period

base = df['booked'].values.astype(float)
naive_booked = np.clip(base + treat*true_user_effect, 0, 1)
# Supply constraint: total bookings capped → treated riders cannibalize controls.
if naive_booked.sum() > supply:
    scale = supply / naive_booked.sum()
    control_booked = base*scale*(1-treat)
    treated_booked = np.clip(base + true_user_effect, 0, 1)*(treat)
else:
    control_booked, treated_booked = base*(1-treat), naive_booked*treat

measured = treated_booked[treat==1].mean() - control_booked[treat==0].mean()
print(f'Measured user-level A/B lift : {measured:+.3f}')
print(f'True per-user effect         : {true_user_effect:+.3f}')
print('→ The user-level estimate is inflated: control bookings were cannibalized by treated demand (SUTVA violated).')

## 3. Heterogeneous effects with EconML (Double ML)

We estimate how the causal effect of surge varies across riders, controlling for the confounder. `LinearDML` partials out covariates from both treatment and outcome, then fits the effect — recovering the heterogeneity the simulator built in.

In [ ]:
try:
    from econml.dml import LinearDML
    from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier

    X = df[gt['feature_cols']].values         # effect modifiers
    W = df[['demand_intensity']].values       # confounders to control
    T = df['surge'].values
    Y = df['booked'].values

    est = LinearDML(
        model_y=GradientBoostingClassifier(),
        model_t=GradientBoostingRegressor(),
        discrete_treatment=False, random_state=0,
    )
    est.fit(Y, T, X=X, W=W)
    cate = est.effect(X)
    print(f'Estimated ATE : {cate.mean():+.3f}   (TRUE {gt["ate"]:+.3f})')

    # Heterogeneity check: commuters should be LESS price sensitive than leisure.
    comm = cate[df['is_commuter']==1].mean(); leis = cate[df['is_commuter']==0].mean()
    print(f'CATE commuters: {comm:+.3f}   CATE leisure: {leis:+.3f}')
    print('→ commuters less negative (less price-sensitive):', comm > leis)
except Exception as e:
    print('EconML section skipped:', type(e).__name__, e)

In [ ]:
# Compare estimated vs. true CATE if EconML ran.
try:
    plt.figure(figsize=(6,5))
    plt.scatter(df['true_cate'], cate, s=6, alpha=0.2, color='#276EF1')
    lims = [df['true_cate'].min(), df['true_cate'].max()]
    plt.plot(lims, lims, 'k--', label='perfect')
    plt.xlabel('true CATE'); plt.ylabel('estimated CATE'); plt.legend()
    plt.title('EconML recovers heterogeneous price sensitivity'); plt.tight_layout(); plt.show()
except NameError:
    print('cate not available (EconML skipped).')

## 4. Assembling a causal pricing structure

Putting the series together, a defensible pricing system for a two-sided marketplace looks like:

1. **Predict demand/value** from features — the model from [notebook 01](01_predictive_pricing.ipynb). Gives a starting point and personalization features.
2. **Estimate the *causal* elasticity** — not from observational fits (notebook 03) but from **experiments / IV**, with **heterogeneous** effects per segment (this notebook, EconML CATE).
3. **Optimize price per segment** using the causal elasticity: pick the price that maximizes revenue *or* welfare, given $\varepsilon(x)$.
4. **Respect the marketplace**: account for **interference** (use market-level/switchback experiments), **both sides** (a price that delights riders but loses drivers fails), and **guardrails** (fairness, price-gouging limits, long-run retention).

### Decision flow

> Is the price variation in my data **randomized**? — **Yes →** estimate effects directly (DML/CATE). **No →** do I have a valid **instrument**? — **Yes →** IV/2SLS. **No →** you cannot trust the elasticity; **run an experiment** before changing prices.

### The one-line takeaway

> A model that predicts the price you *see* is not a model of how demand *responds* to the price you *set*. Pricing is a **causal** problem, and in a two-sided marketplace it is a causal problem **with interference** — design for that from the start.

**Further reading**
- [Uber — Causally-Informed Marketplace Optimization (arXiv:2407.19078)](https://arxiv.org/html/2407.19078v1)
- [Airbnb — Customized Regression Model for Dynamic Pricing (KDD 2018)](https://www.kdd.org/kdd2018/accepted-papers/view/customized-regression-model-for-airbnb-dynamic-pricing)
- [Causal Inference for the Brave and True](https://matheusfacure.github.io/python-causality-handbook/)
- [Causal Inference: The Mixtape](https://mixtape.scunning.com/)